In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install numpy==2.1.3

In [ ]:
!pip install pandas==2.2.3

In [ ]:
!pip install matplotlib==3.9.2

In [ ]:
!pip install tabulate==0.9.0

In [ ]:
!pip freeze

In [ ]:
import re
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

In [ ]:
project_id: int = 0 # @TODO: Set a project ID.

In [ ]:
conn = sqlite3.connect('../database/db.sqlite')

Number of test in-/exclusions

In [ ]:
query = f"SELECT project_id, count(*) AS total_count, sum(is_included) AS included_count FROM test WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

Number of generalization in-/exclusions

In [ ]:
query = f"SELECT project_id, variant, count(*) AS total_count, sum(is_included) AS included_count FROM generalization WHERE project_id = {project_id} GROUP BY project_id, variant"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

Number of test and generalization exclusions per task

In [ ]:
query = f"""
SELECT project_id, 1 AS type_id, 'TEST' AS item_type, 'ORIGINAL' AS variant, exclusion_info FROM test WHERE project_id = {project_id} AND is_included = 0
UNION ALL
SELECT project_id, 2 AS type_id, 'GENERALIZATION', variant, exclusion_info FROM generalization WHERE project_id = {project_id} AND is_included = 0
"""
df = pd.read_sql_query(query, conn)

def extract_task_name(s):
    match = re.search(r'\b(\w+){', s)
    return match.group(1) if match else None

df['exclusion_info'] = df['exclusion_info'].apply(extract_task_name)

df = df.pivot_table(index=['project_id', 'type_id', 'item_type', 'variant'], columns='exclusion_info', aggfunc='size', fill_value=0)
df['Total Exclusions'] = df.sum(axis=1)
df = df[['Total Exclusions'] + [col for col in df if col != 'Total Exclusions']]

df

Number of exclusions caused by task failures


In [ ]:
query = f"SELECT project_id, step, stage, variant, count(*) FROM task WHERE project_id = {project_id} AND status = 'FAILED' GROUP BY project_id, step, stage, variant"
df = pd.read_sql_query(query, conn)
df

Causes of task failure-based exclusions

In [ ]:
query = f"SELECT project_id, step, stage, variant, info FROM task WHERE project_id = {project_id} AND status = 'FAILED'"
df = pd.read_sql_query(query, conn)

failure_types = [
    'Depth limit of 100 exceeded',
    'PC size limit exceeded',
    'Execution timeout exceeded',
    'AssertionFailedError',
    'java.lang.ArithmeticException: !!!div by 0',
    'java.lang.ClassNotFoundException: class not found: java.lang.NoSuchMethodException!!',
    'method arguments do not match with JPF\'s symbolic.method configuration',
    'no peer'
]

def categorize_failure(info):
    for failure_type in failure_types:
        if failure_type in info:
            return failure_type
    return '<Other>'

df['failure_type'] = df['info'].apply(categorize_failure)

failure_causes = df['failure_type'].value_counts().reset_index()
failure_causes = pd.concat([failure_causes, pd.DataFrame({'failure_type': ['<Total>'], 'count': [df['info'].count()]})])
failure_causes.sort_values(by='count', ascending=False).reset_index(drop=True)

#Other failures:
# mask = ~df['info'].str.contains('|'.join(failure_types))
# no_match_df = df[mask]
# no_match_df['info']

Causes of filtering-based test exclusions

In [ ]:
query = f"SELECT project_id, exclusion_info FROM test WHERE project_id = {project_id} AND exclusion_info LIKE '%TestFilteringTask%'"
df = pd.read_sql_query(query, conn)

def process_info(info):
    lines = info.split('\n')
    result = {}
    for line in lines:
        split_line = line.split(': ')
        if len(split_line) == 2:
            result[split_line[0]] = 0 if (split_line[1] == 'ACCEPT') else 1
    return result

df['exclusion_info_dict'] = df['exclusion_info'].apply(process_info)

filter_df = pd.json_normalize(df['exclusion_info_dict'])

df = pd.concat([df, filter_df], axis=1)
df = df.drop(columns=['exclusion_info'])
df = df.drop(columns=['exclusion_info_dict'])

df.groupby('project_id').sum()

Mutation testing results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, status FROM pit_mutation_report WHERE project_id = {project_id}"
df = pd.read_sql_query(query, conn)

df['variant'] = df['variant'].fillna('ORIGINAL')

mutation_status_categories = ['SURVIVED', 'KILLED', 'TIMED_OUT', 'NO_COVERAGE', 'NON_VIABLE', 'MEMORY_ERROR', 'RUN_ERROR']
mutation_status_categories = [c for c in mutation_status_categories if c in df['status'].unique()]
df['status'] = pd.Categorical(df['status'], categories=mutation_status_categories)

df = df.pivot_table(index=['project_id', 'step', 'stage', 'variant'], columns='status', aggfunc='size', fill_value=0, observed=True)
df['TOTAL'] = df.sum(axis=1)
df = df[['TOTAL'] + mutation_status_categories]
df['% killed of covered'] = df['KILLED'] / (df['TOTAL'] - df['NO_COVERAGE'])

df.reset_index(inplace=True)
df

Code coverage results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(instruction_missed), sum(instruction_covered), sum(branch_missed), sum(branch_covered) FROM jacoco_coverage_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant"
df = pd.read_sql_query(query, conn)
df

Runtime requirements per processing stage

In [ ]:
query = f"SELECT project_id, step, stage, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, step, stage ORDER BY project_id, step"
df = pd.read_sql_query(query, conn)
df

In [ ]:
df['step_stage'] = df['step'].astype(str) + "-" + df['stage'].astype(str)

df.plot(kind='bar', x='step_stage', y='runtime', legend=None, figsize=(12, 6))

plt.xlabel('Processing Stage')
plt.ylabel('Runtime (in seconds)')
plt.title('Runtime per Processing Stage')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

Causes of test failures

In [ ]:
query = f"SELECT project_id, step, stage, variant, failure_type, count(*), sum(runtime) FROM junit_test_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant, failure_type"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df['failure_type'] = df['failure_type'].fillna('')
df